# Load modules

In [5]:
import dask_jobqueue
import dask
import dask.distributed as dask_distributed
import os
import xarray as xr
import pyproj
import numpy as np
from gsw import f

import functions

# Open dask-cluster

In [2]:
%%time
cluster = dask_jobqueue.SLURMCluster(cores=4,memory='20GB',processes=1,queue='base', walltime='20:00:00',interface='ib0',local_directory='$TMPDIR')
client = dask.distributed.Client(cluster)
cluster.scale(jobs=6)
client

CPU times: user 517 ms, sys: 691 ms, total: 1.21 s
Wall time: 2.98 s


Connection method: Cluster object,Cluster type: dask_jobqueue.SLURMCluster
Dashboard: http://172.18.4.213:8787/status,
Dashboard: http://172.18.4.213:8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://172.18.4.213:41611,Workers: 0
Dashboard: http://172.18.4.213:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


# Define directories

In [15]:
hcs = 300 # horizontal chunk size
path_data = '/gxfs_work/geomar/smomw355/model_data/ocean-only/INALT60.L120-KRS0020/nemo/'
path_mask = path_data + 'suppl/2_INALT60.L120-KRS0020_mesh_mask.nc'
path_data += 'output/2_INALT60.L120-KRS0020_'

path_current = os.getcwd()
path_save = os.path.join(path_current, 'Fields_INALT60') 
os.makedirs(path_save, exist_ok=True)

path_save_ssh = os.path.join(path_save, 'SSH_filtered') 
path_save_w =  os.path.join(path_save, 'w_filtered')
path_save_OW =  os.path.join(path_save, 'strain_vorticity')

os.makedirs(path_save_ssh, exist_ok=True)
os.makedirs(path_save_w, exist_ok=True)
os.makedirs(path_save_OW, exist_ok=True)

# Define period and region of interest

In [12]:
frq = '1d' #output frequency
yeara = 2012
yearb = 2017
yearvec = np.arange(yeara,yearb+1)
month_array=np.arange(1,13)

lon_c=15.7
lat_c=-37.8
pos_west,pos_east,pos_south,pos_north  = lon_c-2,lon_c+2,lat_c-2,lat_c+2

# Read data

In [11]:
hcs = 300 # horizontal chunk size
path_data = '/gxfs_work/geomar/smomw355/model_data/ocean-only/INALT60.L120-KRS0020/nemo/'
path_mask = path_data + 'suppl/2_INALT60.L120-KRS0020_mesh_mask.nc'
path_data += 'output/2_INALT60.L120-KRS0020_'

ds_mask = xr.open_dataset(path_mask,chunks={'x':hcs,'y':hcs})
ds_mask_region = ds_mask.where((ds_mask.nav_lat>pos_south-1.2) & (ds_mask.nav_lat<pos_north+1.2) & (ds_mask.nav_lon>pos_west-1.2) & (ds_mask.nav_lon<pos_east+1.2), drop=True)

# Define a regular metric grid (needed for the eSQG comparison) for interpolation

In [13]:
dx=2 #regular spacing in km
lon_grid=ds_mask_region.isel(z=0).nav_lon
lat_grid=ds_mask_region.isel(z=0).nav_lat

#Define the grid
a = pyproj.Transformer.from_crs(4326,3395).transform(lon_grid,lat_grid)
y3 = a[0]
x3 = a[1]
x3min = np.nanmin(x3,1)
y3min = np.nanmin(y3,0)
Y3min,X3min = np.meshgrid(y3min, x3min)
x1=(x3-X3min)/1000
y1=(y3-Y3min)/1000
x_len = int(np.floor(np.amax(x1))+1)
y_len = int(np.floor(np.amax(y1))+1)
x_dim = np.linspace(0, x_len, int(x_len/dx), endpoint=False)
y_dim = np.linspace(0, y_len, int(y_len/dx), endpoint=False)
x2, y2 = np.meshgrid(x_dim, y_dim)

#Retrieve longitudes and latitudes back from the interpolation 
x2_proj = x2*1000 + x3min[0]
y2_proj = y2*1000 + y3min[0]
transformer_back = pyproj.Transformer.from_crs(3395, 4326, always_xy=True)
lat2, lon2 = transformer_back.transform(y2_proj,x2_proj)

#Make square
size_x=len(lon2[0,:])
size_y=len(lon2[:,0])

if size_y>size_x:
    delta_points=size_y-size_x
    lon2 = lon2[:size_x,:]
    lat2 = lat2[:size_x,:]
else:
    delta_points=size_x-size_y
    lon2 = lon2[:,:size_y]
    lat2 = lat2[:,:size_y]

# Loop over each year and each month to compute and save:
#### 1) SSH fields filtered
#### 2) w fields filtered
#### 3) strain and vorticity at three different depths

In [ ]:
for yeara in yearvec:
    #Open files
        #SSH
    paths = !ls {path_data + frq + '_' + str(yeara) + '*' + '_grid_T.nc'}
    ds_ssh = xr.open_mfdataset(paths,chunks={'x':hcs,'y':hcs,'depthv':1,'time_counter':-1}).squeeze().rename({'deptht':'z'})
        #Zonal velocity
    paths = !ls {path_data + frq + '_' + str(yeara) + '*' + '_grid_U.nc'}
    ds_u = xr.open_mfdataset(paths,chunks={'x':hcs,'y':hcs,'depthu':1,'time_counter':-1}).squeeze().rename({'depthu':'z'})
        #Meridional velocity
    paths = !ls {path_data + frq + '_' + str(yeara) + '*' + '_grid_V.nc'}
    ds_v = xr.open_mfdataset(paths,chunks={'x':hcs,'y':hcs,'depthv':1,'time_counter':-1}).squeeze().rename({'depthv':'z'})  
        #Vertical velocity
    paths = !ls {path_data + frq + '_' + str(yeara) + '*' + '_grid_W.nc'}
    ds_w = xr.open_mfdataset(paths,chunks={'x':hcs,'y':hcs,'depthw':1,'time_counter':-1}).squeeze().rename({'depthw':'z'})
    
    #Resize dataset to the region of interest
    ds_ssh_region = ds_ssh.where((ds_mask.nav_lat>pos_south-1.2) & (ds_mask.nav_lat<pos_north+1.2) & (ds_mask.nav_lon>pos_west-1.2) & (ds_mask.nav_lon<pos_east+1.2), drop=True)
    ds_w_region = ds_w.where((ds_mask.nav_lat>pos_south-1.2) & (ds_mask.nav_lat<pos_north+1.2) & (ds_mask.nav_lon>pos_west-1.2) & (ds_mask.nav_lon<pos_east+1.2), drop=True)
    ds_u_region = ds_u.where((ds_mask.nav_lat>pos_south-1.2) & (ds_mask.nav_lat<pos_north+1.2) & (ds_mask.nav_lon>pos_west-1.2) & (ds_mask.nav_lon<pos_east+1.2), drop=True)
    ds_v_region = ds_v.where((ds_mask.nav_lat>pos_south-1.2) & (ds_mask.nav_lat<pos_north+1.2) & (ds_mask.nav_lon>pos_west-1.2) & (ds_mask.nav_lon<pos_east+1.2), drop=True)

    #Cut depth
    mask_1000 = (ds_u_region.z <= 650) & (ds_u_region.z >= 200)
    ds_mask_region = ds_mask_region.where(mask_1000, drop=True)
    
    mask_1000_w = (ds_w_region.z <= 700) & (ds_w_region.z >= 150)
    ds_w_region = ds_w_region.where(mask_1000_w, drop=True)

    #Loop over each month
    for m in month_array:
        
        #Select month of interest
        ds_ssh_region_m = ds_ssh_region.where(ds_ssh_region.time_counter.dt.month.isin([m]), drop=True)
        ds_w_region_m = ds_w_region.where(ds_w_region.time_counter.dt.month.isin([m]), drop=True)
        ds_u_region_m = ds_u_region.where(ds_u_region.time_counter.dt.month.isin([m]), drop=True)
        ds_v_region_m = ds_v_region.where(ds_v_region.time_counter.dt.month.isin([m]), drop=True)

        #Interpolate on a regular grid - SSH
        ssh_model = ds_ssh_region_m.sossheig.where(ds_mask_region.tmask.isel(z=0)==1).squeeze() 

        ssh_gridded = xr.apply_ufunc(interp_to_grid,
                 ssh_model.chunk(
        {"x": -1, "y": -1, "time_counter": 7}), x1, y1, x2, y2,
                 input_core_dims=[["y","x"], ["y","x"], ["y","x"], ["new_y","new_x"], ["new_y","new_x"]],
                 output_core_dims=[["new_y","new_x"]],
                 exclude_dims=set(("y","x")),
                 vectorize = True,
                 dask="parallelized",
                 output_dtypes=[ssh_model.dtype]).rename({"new_y": "y", "new_x": "x"})

        #Interpolate w on the vertical at SSH grid points - W
        w_org = ds_w_region_m.vovecrtz  
        z1=ds_w_region_m.z
        z2=ds_ssh_region_m.z.where(mask_1000, drop=True).rename({"z": "z_new"})
        
        w_gridded_vert = xr.apply_ufunc(interp_to_prof,
                             w_org.chunk(
                {"x": 100, "y": 100, "z":-1,'time_counter':7}), z1, z2,
                             input_core_dims=[["z"], ["z"], ["z_new"]],
                             output_core_dims=[["z_new"]],
                             exclude_dims=set(("z")),
                             vectorize = True,
                             dask="parallelized",
                             output_dtypes=[w_org.dtype]).rename({"z_new": "z"})
        #Masked data
        w_gridded_vert=w_gridded_vert.where(ds_mask_region.tmask==1).squeeze()*60*60*24

        #Interpolate w on the horizontal on the regular 2-km grid - W
        w_gridded = xr.apply_ufunc(interp_to_grid,
                             w_gridded_vert.isel(z=[0,11,-1]).chunk(
                {"x": -1, "y": -1, "z":1,'time_counter':7}), x1, y1, x2, y2,
                             input_core_dims=[["y","x"], ["y","x"], ["y","x"], ["new_y","new_x"], ["new_y","new_x"]],
                             output_core_dims=[["new_y","new_x"]],
                             exclude_dims=set(("y","x")),
                             vectorize = True,
                             dask="parallelized",
                             output_dtypes=[w_gridded_vert.dtype]).rename({"new_y": "y", "new_x": "x"})

        #Interpolate on a regular grid - U & V
        u_model = ds_u_region_m.vozocrtx.where(ds_mask_region.umask==1).squeeze().isel(z=[0,11,-1])        
        v_model = ds_v_region_m.vomecrty.where(ds_mask_region.vmask==1).squeeze().isel(z=[0,11,-1])     
        
        u_gridded = xr.apply_ufunc(interp_to_grid,
                             u_model.chunk(
                {"x": -1, "y": -1, "z":1, "time_counter": 7}), x1, y1, x2, y2,
                             input_core_dims=[["y","x"], ["y","x"], ["y","x"], ["new_y","new_x"], ["new_y","new_x"]],
                             output_core_dims=[["new_y","new_x"]],
                             exclude_dims=set(("y","x")),
                             vectorize = True,
                             dask="parallelized",
                             output_dtypes=[u_model.dtype]).rename({"new_y": "y", "new_x": "x"})
            
        v_gridded = xr.apply_ufunc(interp_to_grid,
                     v_model.chunk(
        {"x": -1, "y": -1, "z":1, "time_counter": 7}), x1, y1, x2, y2,
                     input_core_dims=[["y","x"], ["y","x"], ["y","x"], ["new_y","new_x"], ["new_y","new_x"]],
                     output_core_dims=[["new_y","new_x"]],
                     exclude_dims=set(("y","x")),
                     vectorize = True,
                     dask="parallelized",
                     output_dtypes=[v_model.dtype]).rename({"new_y": "y", "new_x": "x"})


        #Filter at 40 km and remove 50 grid points (about 100 km) in each direction
        ssh_filtered = filtering_lanczos(ssh_gridded, 40, method).isel(x=slice(50,-50),y=slice(50,-50)).compute()
        w_filtered = filtering_lanczos(w_gridded, 40, method).isel(x=slice(50,-50),y=slice(50,-50)).compute()

        u_filtered = filtering_lanczos(u_gridded, 40)
        v_filtered = filtering_lanczos(v_gridded, 40)

        #Compute vorticity from u and v
        dudy = (u_filtered.shift(y=-1) - u_filtered.shift(y=1)) / (2*1000*dx)
        dvdx = (v_filtered.shift(x=-1) - v_filtered.shift(x=1)) / (2*1000*dx) 
        
        zeta_model=dvdx-dudy
        zeta_model = zeta_model.isel(x=slice(50,-50),y=slice(50,-50)).compute()

        dudx = (u_filtered.shift(x=-1) - u_filtered.shift(x=1)) / (2*1000*dx)
        dvdy = (v_filtered.shift(y=-1) - v_filtered.shift(y=1)) / (2*1000*dx) 
        strain_model = ((dudx-dvdy)**2 + (dudy+dvdx)**2)**.5
        strain_model = strain_model.isel(x=slice(50,-50),y=slice(50,-50)).compute()

        fcoriolis = abs(np.mean(f(lat2[50:-50,50:-50])))

        #Save SSH dataset
        ds_to_save = xr.Dataset(
            data_vars=dict(
                ssh_model=(["time_counter", "y", "x"], ssh_filtered.values)
            ),
            coords=dict(
                time_counter=("time_counter", ssh_filtered.time_counter.values),
                y=("y", lat2[50:-50,50:-50][:,0]),
                x=("x", lon2[50:-50,50:-50][0,:]) 
            ) 
        )
        ds_to_save.to_netcdf(path=path_save_ssh+str(yeara)+'_'+str(m)+'.nc')

        #Save w dataset
        ds_to_save = xr.Dataset(
            data_vars=dict(
                w_model=(["time_counter", "y", "x"], w_filtered.isel(z=1).values)
            ),
            coords=dict(
                time_counter=("time_counter", w_filtered.time_counter.values),
                y=("y", lat2[50:-50,50:-50][:,0]),
                x=("x", lon2[50:-50,50:-50][0,:])
            ) 
        )
        ds_to_save.to_netcdf(path=path_save_w+str(yeara)+'_'+str(m)+'.nc')

        #Save strain, vorticity and w
        ds_to_save = xr.Dataset(
            data_vars=dict(
                w=(["time_counter", "z", "y", "x"], w_filtered.values),
                strain=(["time_counter", "z", "y", "x"], strain_model.values/fcoriolis),
                vorticity=(["time_counter", "z", "y", "x"], zeta_model.values/fcoriolis)
            ),
            coords=dict(
                z=("z", w_filtered.z.values),
                x=("x", w_filtered.x.values),
                y=("y", w_filtered.y.values),
                time_counter=(["time_counter"], ds_w_region_m.time_counter.values)
            ) 
        )
        ds_to_save.to_netcdf(path=path_save+str(yeara)+'_'+str(m)+'.nc')



In [ ]:
for yeara in yearvec:

    paths = !ls {path_data + frq + '_' + str(yeara) + '*' + '_grid_U.nc'}
    ds_u = xr.open_mfdataset(paths,chunks={'x':hcs,'y':hcs,'depthu':1,'time_counter':-1}).squeeze().rename({'depthu':'z'})
    paths = !ls {path_data + frq + '_' + str(yeara) + '*' + '_grid_V.nc'}
    ds_v = xr.open_mfdataset(paths,chunks={'x':hcs,'y':hcs,'depthv':1,'time_counter':-1}).squeeze().rename({'depthv':'z'})  
    paths = !ls {path_data + frq + '_' + str(yeara) + '*' + '_grid_W.nc'}
    ds_w = xr.open_mfdataset(paths,chunks={'x':hcs,'y':hcs,'depthw':1,'time_counter':-1}).squeeze().rename({'depthw':'z'})
    
    #Cut region
    ds_w_region = ds_w.where((ds_mask.nav_lat>pos_south-1.2) & (ds_mask.nav_lat<pos_north+1.2) & (ds_mask.nav_lon>pos_west-1.2) & (ds_mask.nav_lon<pos_east+1.2), drop=True)
    ds_u_region = ds_u.where((ds_mask.nav_lat>pos_south-1.2) & (ds_mask.nav_lat<pos_north+1.2) & (ds_mask.nav_lon>pos_west-1.2) & (ds_mask.nav_lon<pos_east+1.2), drop=True)
    ds_v_region = ds_v.where((ds_mask.nav_lat>pos_south-1.2) & (ds_mask.nav_lat<pos_north+1.2) & (ds_mask.nav_lon>pos_west-1.2) & (ds_mask.nav_lon<pos_east+1.2), drop=True)
    ds_mask_region = ds_mask.where((ds_mask.nav_lat>pos_south-1.2) & (ds_mask.nav_lat<pos_north+1.2) & (ds_mask.nav_lon>pos_west-1.2) & (ds_mask.nav_lon<pos_east+1.2), drop=True)

    #Cut depth
    mask_1000 = (ds_u_region.z <= 650) & (ds_u_region.z >= 200)
    ds_mask_region = ds_mask_region.where(mask_1000, drop=True)
    
    mask_1000_w = (ds_w_region.z <= 700) & (ds_w_region.z >= 150)
    ds_w_region = ds_w_region.where(mask_1000_w, drop=True)

    dx=2
    lon_grid=ds_w_region.nav_lon
    lat_grid=ds_w_region.nav_lat
    
    #Define the grid
    a = pyproj.Transformer.from_crs(4326,3395).transform(lon_grid,lat_grid) # project WGS84 onto metric grid
    y3 = a[0]
    x3 = a[1]
    x3min = np.nanmin(x3,1)
    y3min = np.nanmin(y3,0)
    Y3min,X3min = np.meshgrid(y3min, x3min)
    x1=(x3-X3min)/1000
    y1=(y3-Y3min)/1000
    x_len = int(np.floor(np.amax(x1))+1)
    y_len = int(np.floor(np.amax(y1))+1)
    x_dim = np.linspace(0, x_len, int(x_len/dx), endpoint=False)
    y_dim = np.linspace(0, y_len, int(y_len/dx), endpoint=False)
    x2, y2 = np.meshgrid(x_dim, y_dim)

        #Add nav_lon and nav_lat back from the interpolation 
    x2_proj = x2*1000 + x3min[0]
    y2_proj = y2*1000 + y3min[0]
    transformer_back = pyproj.Transformer.from_crs(3395, 4326, always_xy=True)
    lat2, lon2 = transformer_back.transform(y2_proj,x2_proj)
    
    #Make square - for SQG
    size_x=len(lon2[0,:])
    size_y=len(lon2[:,0])
    
    if size_y>size_x:
        delta_points=size_y-size_x
        lon2 = lon2[:size_x,:]
        lat2 = lat2[:size_x,:]
    else:
        delta_points=size_x-size_y
        lon2 = lon2[:,:size_y]
        lat2 = lat2[:,:size_y]

    #Loop on months
    for m in month_array:

        ds_w_region_m = ds_w_region.where(ds_w_region.time_counter.dt.month.isin([m]), drop=True)
        ds_u_region_m = ds_u_region.where(ds_u_region.time_counter.dt.month.isin([m]), drop=True)
        ds_v_region_m = ds_v_region.where(ds_v_region.time_counter.dt.month.isin([m]), drop=True)

        ##Interpolate on the vertical at T points
        w_org = ds_w_region_m.vovecrtz  
        z1=ds_w_region_m.z
        z2=ds_u_region_m.z.where(mask_1000, drop=True).rename({"z": "z_new"})
        
        w_gridded_vert = xr.apply_ufunc(interp_to_prof,
                             w_org.chunk(
                {"x": 100, "y": 100, "z":-1,'time_counter':7}), z1, z2,
                             input_core_dims=[["z"], ["z"], ["z_new"]],
                             output_core_dims=[["z_new"]],
                             exclude_dims=set(("z")),
                             vectorize = True,
                             dask="parallelized",
                             output_dtypes=[w_org.dtype]).rename({"z_new": "z"})
        #Masked data
        w_gridded_vert=w_gridded_vert.where(ds_mask_region.tmask==1).squeeze()*60*60*24
        
        w_gridded = xr.apply_ufunc(interp_to_grid,
                             w_gridded_vert.isel(z=[0,11,-1]).chunk(
                {"x": -1, "y": -1, "z":1,'time_counter':7}), x1, y1, x2, y2,
                             input_core_dims=[["y","x"], ["y","x"], ["y","x"], ["new_y","new_x"], ["new_y","new_x"]],
                             output_core_dims=[["new_y","new_x"]],
                             exclude_dims=set(("y","x")),
                             vectorize = True,
                             dask="parallelized",
                             output_dtypes=[w_gridded_vert.dtype]).rename({"new_y": "y", "new_x": "x"})

                # Re-grid + filter
        u_model = ds_u_region_m.vozocrtx.where(ds_mask_region.umask==1).squeeze().isel(z=[0,11,-1])        
        v_model = ds_v_region_m.vomecrty.where(ds_mask_region.vmask==1).squeeze().isel(z=[0,11,-1])     
        
        u_gridded = xr.apply_ufunc(interp_to_grid,
                             u_model.chunk(
                {"x": -1, "y": -1, "z":1, "time_counter": 7}), x1, y1, x2, y2,
                             input_core_dims=[["y","x"], ["y","x"], ["y","x"], ["new_y","new_x"], ["new_y","new_x"]],
                             output_core_dims=[["new_y","new_x"]],
                             exclude_dims=set(("y","x")),
                             vectorize = True,
                             dask="parallelized",
                             output_dtypes=[u_model.dtype]).rename({"new_y": "y", "new_x": "x"})
            
        v_gridded = xr.apply_ufunc(interp_to_grid,
                     v_model.chunk(
        {"x": -1, "y": -1, "z":1, "time_counter": 7}), x1, y1, x2, y2,
                     input_core_dims=[["y","x"], ["y","x"], ["y","x"], ["new_y","new_x"], ["new_y","new_x"]],
                     output_core_dims=[["new_y","new_x"]],
                     exclude_dims=set(("y","x")),
                     vectorize = True,
                     dask="parallelized",
                     output_dtypes=[v_model.dtype]).rename({"new_y": "y", "new_x": "x"})

        ##Filter - 20 km
        w_filtered = filtering_lanczos(w_gridded, 40).isel(x=slice(50,-50),y=slice(50,-50)).compute()
        u_filtered = filtering_lanczos(u_gridded, 40)
        v_filtered = filtering_lanczos(v_gridded, 40)

        #Compute vorticity from u and v
        dudy = (u_filtered.shift(y=-1) - u_filtered.shift(y=1)) / (2*1000*dx)
        dvdx = (v_filtered.shift(x=-1) - v_filtered.shift(x=1)) / (2*1000*dx) 
        
        zeta_model=dvdx-dudy
        zeta_model = zeta_model.isel(x=slice(50,-50),y=slice(50,-50)).compute()

        dudx = (u_filtered.shift(x=-1) - u_filtered.shift(x=1)) / (2*1000*dx)
        dvdy = (v_filtered.shift(y=-1) - v_filtered.shift(y=1)) / (2*1000*dx) 
        strain_model = ((dudx-dvdy)**2 + (dudy+dvdx)**2)**.5
        strain_model = strain_model.isel(x=slice(50,-50),y=slice(50,-50)).compute()

        fcoriolis = abs(np.mean(f(lat2[50:-50,50:-50])))

        ds_to_save = xr.Dataset(
            data_vars=dict(
                w=(["time_counter", "z", "y", "x"], w_filtered.values),
                strain=(["time_counter", "z", "y", "x"], strain_model.values/fcoriolis),
                vorticity=(["time_counter", "z", "y", "x"], zeta_model.values/fcoriolis)
            ),
            coords=dict(
                z=("z", w_filtered.z.values),
                x=("x", w_filtered.x.values),
                y=("y", w_filtered.y.values),
                time_counter=(["time_counter"], ds_w_region_m.time_counter.values)
            ) 
        )
        ds_to_save.to_netcdf(path=path_save_OW+str(yeara)+'_'+str(m)+'.nc')

        print('File saved')